# Notebook that Orchestrates the Forecasting pipeline



In [1]:
## bootstrap setup for setup on different environments (Colab, Kaggle, local)

In [2]:
from pathlib import Path
import os
import sys
import subprocess

def in_colab() -> bool:
    return "google.colab" in sys.modules

def in_kaggle() -> bool:
    return os.path.exists("/kaggle")

def find_repo_root(start: Path) -> Path:
    """
    Walk upwards until we find a directory that looks like the repo root.
    """
    current = start.resolve()
    markers = ["src", "data", "notebooks", "requirements.txt", ".gitignore"]

    for candidate in [current] + list(current.parents):
        if all((candidate / m).exists() for m in ["src", "data", "notebooks"]):
            return candidate

    raise FileNotFoundError(
        f"Could not locate repo root starting from: {start}"
    )

print("Environment:",
      "Colab" if in_colab() else "Kaggle" if in_kaggle() else "Local")

REPO_NAME = "Clustering-And-Forecasting-TimeSeries-PlayingGround"
REPO_URL = "https://github.com/QuirkyCroissant/Clustering-And-Forecasting-TimeSeries-PlayingGround.git"

if in_colab() or in_kaggle():
    base_dir = Path("/kaggle/working") if in_kaggle() else Path("/content")
    repo_dir = base_dir / REPO_NAME

    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

    os.chdir(repo_dir)
    REPO_ROOT = repo_dir.resolve()
else:
    REPO_ROOT = find_repo_root(Path.cwd())

PROJECT_ROOT = REPO_ROOT

SRC_PATH = REPO_ROOT / "src"
if not SRC_PATH.exists():
    raise FileNotFoundError(f"src path not found: {SRC_PATH}")

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH:", SRC_PATH)
print("sys.path[0]:", sys.path[0])
print("forecasting dir exists:", (SRC_PATH / "forecasting").exists())

Environment: Local
REPO_ROOT: /home/gandalf/Documents/github-repos/Clustering-And-Forecasting-TimeSeries-PlayingGround
SRC_PATH: /home/gandalf/Documents/github-repos/Clustering-And-Forecasting-TimeSeries-PlayingGround/src
sys.path[0]: /usr/lib/python313.zip
forecasting dir exists: True


In [3]:
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

OUTPUT_METRICS = PROJECT_ROOT / "outputs" / "metrics"
OUTPUT_PLOTS = PROJECT_ROOT / "outputs" / "plots"
OUTPUT_PREDS = PROJECT_ROOT / "outputs" / "predictions"
OUTPUT_MODELS = PROJECT_ROOT / "outputs" / "models"

for p in [OUTPUT_METRICS, OUTPUT_PLOTS, OUTPUT_PREDS, OUTPUT_MODELS]:
    p.mkdir(parents=True, exist_ok=True)

## Importing python modules and helper functions

In [4]:
import pandas as pd
from forecasting.data import load_wide_csv, load_cluster_labels, ensure_output_dirs
from forecasting.features import make_training_frame
from forecasting.train import fit_global_model, fit_cluster_models
from forecasting.predict import forecast_global, forecast_by_group
from forecasting.evaluate import mae_by_household, summarise_mae

OUT = ensure_output_dirs(PROJECT_ROOT)

TRAIN_23_PATH = PROJECT_ROOT / "data" / "raw" / "sample_23.csv"
TEST_24_PATH  = PROJECT_ROOT / "data" / "raw" / "sample_24.csv"

# Start with one clustering output first.
# If your clustering notebook saved under notebooks/outputs/feature:
CLUSTER_PATH = PROJECT_ROOT / "notebooks" / "outputs" / "feature" / "case2_clusters.csv"

In [5]:
train_23 = load_wide_csv(TRAIN_23_PATH)
test_24 = load_wide_csv(TEST_24_PATH)
cluster_labels = load_cluster_labels(CLUSTER_PATH)

# --- Debug subsampling and progress setup ---------------------------------
# When debugging large pipelines, keep only a small fraction of rows to
# reduce memory/compute and avoid IDE variable-inspector timeouts.
DEBUG = True
DEBUG_FRAC = 0.2  # use 20% of the data for quick debugging

from tqdm.auto import tqdm
# enable pandas progress_apply / progress_map if used downstream
tqdm.pandas()

if DEBUG:
    # assume the first column in the wide frames is the ID column
    id_col_train = train_23.columns[0]
    id_col_clusters = cluster_labels.columns[0] if len(cluster_labels.columns) > 0 else id_col_train
    sampled_ids = train_23[id_col_train].sample(frac=DEBUG_FRAC, random_state=42)

    train_23 = train_23[train_23[id_col_train].isin(sampled_ids)].reset_index(drop=True)
    test_24 = test_24[test_24[id_col_train].isin(sampled_ids)].reset_index(drop=True)
    cluster_labels = cluster_labels[cluster_labels[id_col_clusters].isin(sampled_ids)].reset_index(drop=True)

    print(f"DEBUG MODE: using {int(DEBUG_FRAC*100)}% of data -> ")
    print("  train_23.shape:", train_23.shape)
    print("  test_24.shape:", test_24.shape)
    print("  cluster_labels.shape:", cluster_labels.shape)


train_df = make_training_frame(
    train_wide=train_23,
    cluster_labels=cluster_labels,
    static_features=None,   # add shapelet/static features later
)

feature_cols = [c for c in train_df.columns if c not in ["ID", "ForecastGroup", "target"]]
future_dates = pd.to_datetime(test_24.columns[1:])

print(train_df.shape)
print(feature_cols[:10])

DEBUG MODE: using 20% of data -> 
  train_23.shape: (3509, 366)
  test_24.shape: (3509, 367)
  cluster_labels.shape: (3509, 3)
(1182533, 29)
['dow', 'month', 'doy', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_7', 'lag_14']


## Fitting Models to data

In [6]:
global_lgbm = fit_global_model(train_df, feature_cols, model_name="lgbm")
cluster_lgbm_models = fit_cluster_models(train_df, feature_cols, model_name="lgbm", min_rows=500)

ModuleNotFoundError: No module named 'lightgbm'

In [ ]:
pred_global_lgbm = forecast_global(
    train_23_wide=train_23,
    future_dates=future_dates,
    model=global_lgbm,
)

In [ ]:
pred_cluster_lgbm = forecast_by_group(
    train_23_wide=train_23,
    cluster_labels=cluster_labels,
    future_dates=future_dates,
    group_models=cluster_lgbm_models,
    fallback_model=global_lgbm,
)

### Evaluating and comparing results

In [ ]:
mae_global = mae_by_household(pred_global_lgbm, test_24)
mae_cluster = mae_by_household(pred_cluster_lgbm, test_24)

summary_global = summarise_mae(mae_global)
summary_global["model"] = "global_lgbm"

summary_cluster = summarise_mae(mae_cluster)
summary_cluster["model"] = "cluster_lgbm"

summary = pd.concat([summary_global, summary_cluster], ignore_index=True)
display(summary)

pred_global_lgbm.to_csv(OUT["predictions"] / "pred_2024_global_lgbm.csv", index=False)
pred_cluster_lgbm.to_csv(OUT["predictions"] / "pred_2024_cluster_lgbm.csv", index=False)
summary.to_csv(OUT["metrics"] / "mae_summary_lgbm.csv", index=False)
mae_global.to_csv(OUT["metrics"] / "mae_by_household_global_lgbm.csv", index=False)
mae_cluster.to_csv(OUT["metrics"] / "mae_by_household_cluster_lgbm.csv", index=False)